# 01. Análise exploratória (EDA)

**Objetivo:** diagnosticar as quatro bases brutas (estrutura, nulos, duplicatas, outliers, distribuições e comportamento no tempo) sem usar inadimplência. Aqui só se diagnostica; o tratamento fica para outro notebook.

In [ ]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

import src.functions as fn #Funcoes criadas no projeto
import pandas as pd
import numpy as np

In [ ]:
# Leitura das bases e criação de cópias para tratamento.
cadastral_bruta = pd.read_parquet("../dados/base_cadastral.parquet")
cadastral = cadastral_bruta.copy()
submissao_bruta = pd.read_parquet("../dados/base_submissao.parquet")
submissao = submissao_bruta.copy()
emprestimos_bruta = pd.read_parquet("../dados/historico_emprestimos.parquet")
emprestimos = emprestimos_bruta.copy()
parcelas_bruta = pd.read_parquet("../dados/historico_parcelas.parquet")
parcelas = parcelas_bruta.copy()

for nome, df in [("cadastral", cadastral_bruta), ("submissao", submissao_bruta),
                 ("emprestimos", emprestimos_bruta), ("parcelas", parcelas_bruta)]:
    print(f"{nome}: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

## Base Cadastral

In [ ]:
cadastral.info()

In [ ]:
cadastral.head()

In [ ]:
# Aplicando função criada para fazer uma auditoria profunda em dados categóricos (excluindo colunas com tipos incorretos).
resumo, freq, variantes = fn.auditar_categoricas(
    cadastral, excluir=["data_nascimento", "renda_anual"]
)
resumo

In [ ]:
freq

In [ ]:
variantes

In [ ]:
cadastral.describe()

In [ ]:
# Avaliando estrutura de colunas que deveriam ter tipos diferents:
print(cadastral["data_nascimento"].str.replace(r"\d", "9", regex=True).value_counts(dropna=False).head(10))
print(cadastral["renda_anual"].str.replace(r"\d", "", regex=True).value_counts(dropna=False).head(10))
cadastral[["data_nascimento", "renda_anual"]].sample(8, random_state=42)

### Decisões para `cadastral` (variáveis categóricas)

- `sexo`, `tipo_renda`, `possui_carro`, `possui_imovel`: padronizar caixa, acentos e espaços para um rótulo canônico (Feminino, Masculino, Sim, Não, Assalariado, Autônomo).
- Placeholder "Não informado" em `sexo`, `possui_carro` e `possui_imovel` (~0,1% cada): converter para nulo e decidir a imputação no pipeline, ajustada só no treino.
- `nivel_educacao`, `tipo_moradia`, `ocupacao` (categorias): sem variantes de digitação.
- Pendentes: tratamento do placeholder em `tipo_renda`, `tipo_organizacao` e `estado_civil`; significado dos nulos de `ocupacao`; categorias raras.
- Há colunas que deverão ser excluídas antes do merge com as demais, como `sexo` (dados sensíveis) e `qtd_filhos` (redundante)

### Decisões de tipos para `cadastral`

- `data_nascimento`: dois formatos, `AAAA-MM-DD` (50.562) e `DD/MM/AAAA` (38, ordem confirmada porque o 1º campo passa de 12). Converter com `["%Y-%m-%d", "%d/%m/%Y"]`.
- `renda_anual`: texto com ponto de milhar e vírgula decimal, 101 valores com prefixo `R$` e 101 com "não informado" (viram nulo). Converter com `decimal=","` e `milhar="."`.
- `renda_anual`: 67 valores negativos, a tratar como inconsistência depois da conversão.
- `qtd_membros_familia`: float, converter para `Int64` se não houver valores fracionários.

## Base Submissao

In [ ]:
submissao.info()

In [ ]:
submissao.head()

In [ ]:
resumo, frequencias, variantes = fn.auditar_categoricas(submissao, colunas=["tipo_contrato"])

In [ ]:
resumo.T

In [ ]:
frequencias

In [ ]:
variantes

In [ ]:
# Aplica a padronização só em tipo_contrato.
submissao, rel = fn.padronizar_categoricas(submissao, colunas=["tipo_contrato"])
print(rel.T)

In [ ]:
# Checagens: 3 categorias, mesma contagem de linhas e total por categoria igual ao da auditoria.
print(submissao["tipo_contrato"].value_counts(dropna=False))
print("linhas:", len(submissao))

In [ ]:
# Quantos nulos e zeros existem em valor_parcela.
print("nulos:", submissao["valor_parcela"].isna().sum())
print("zeros:", (submissao["valor_parcela"] == 0).sum())
print("negativos:", (submissao["valor_parcela"] < 0).sum())

# Distribuição das parcelas positivas, para saber o que é "normal".
print(submissao.loc[submissao["valor_parcela"] > 0, "valor_parcela"].describe())

# As linhas com zero: o crédito delas é positivo? Em que tipos de contrato aparecem?
zeros = submissao[submissao["valor_parcela"] == 0]
print("\ncrédito nas linhas com parcela zero:")
print(zeros["valor_credito"].describe())
print(zeros["tipo_contrato"].value_counts())

# Os 5 nulos, para inspeção direta.
print(submissao[submissao["valor_parcela"].isna()])

In [ ]:
nulos_antes = submissao["valor_parcela"].isna().sum()

# Parcela zero com crédito positivo é inconsistente: trocamos por nulo.
# .replace(0, np.nan) só afeta valores exatamente iguais a zero.
submissao["valor_parcela"] = submissao["valor_parcela"].replace(0, np.nan)

# Checagens: esperamos 5 -> 72 nulos, 0 zeros e a mesma quantidade de linhas.
print("nulos antes:", nulos_antes, "| depois:", submissao["valor_parcela"].isna().sum())
print("zeros restantes:", (submissao["valor_parcela"] == 0).sum())
print("linhas:", len(submissao))

In [ ]:
# Nulos, zeros e negativos em valor_bem.
print("nulos:", submissao["valor_bem"].isna().sum())
print("zeros:", (submissao["valor_bem"] == 0).sum())
print("negativos:", (submissao["valor_bem"] < 0).sum())

# Distribuição dos valores preenchidos, para saber o que é "normal".
print(submissao["valor_bem"].describe())

# Coluna auxiliar: 1 quando valor_bem é nulo, 0 caso contrário.
# A média dessa coluna dentro de cada grupo é exatamente a proporção de nulos.
bem_nulo = np.where(submissao["valor_bem"].isna(), 1, 0)

# Taxa de nulos por tipo de contrato: se for parecida, a ausência não depende do produto.
print(pd.Series(bem_nulo, index=submissao.index).groupby(submissao["tipo_contrato"]).mean())

# Os nulos têm crédito e parcela diferentes dos demais? Compara as medianas.
print(submissao.groupby(submissao["valor_bem"].isna())[["valor_credito", "valor_parcela"]].median())

In [ ]:
# Flag de ausência: 1 quando valor_bem é nulo, 0 caso contrário.
submissao["valor_bem_ausente"] = np.where(submissao["valor_bem"].isna(), 1, 0)

# Distribuição do crédito em cada grupo: se os intervalos mal se sobrepõem,
# existe uma regra de negócio por trás da ausência.
print(submissao.groupby("valor_bem_ausente")["valor_credito"].describe())

# Checagens: 2.460 flags iguais a 1 e nenhuma linha perdida.
print("flag = 1:", submissao["valor_bem_ausente"].sum())
print("linhas:", len(submissao))

In [ ]:
# Razão crédito/bem, calculada só onde o bem existe e é positivo.
razao = np.where(submissao["valor_bem"] > 0, submissao["valor_credito"] / submissao["valor_bem"], np.nan)
submissao["razao_credito_bem"] = razao

# Distribuição da razão: o esperado é concentrar perto de 1, com uma cauda isolada em ~100.
print(submissao["razao_credito_bem"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99, 0.999]))

# Quantas linhas caem em cada faixa.
faixas = pd.cut(submissao["razao_credito_bem"], bins=[0, 0.5, 1.5, 5, 80, 200, np.inf])
print(faixas.value_counts(sort=False))

# As maiores razões, com os valores originais, para inspeção direta.
print(submissao.nlargest(10, "valor_credito")[["id_cliente", "tipo_contrato", "valor_credito", "valor_bem", "valor_parcela", "razao_credito_bem"]])

In [ ]:
print(list(emprestimos.columns))
# Seleciona os 63 contratos suspeitos pela mesma regra de razão (80 a 200), sem alterar nada.
suspeitos = submissao[(submissao["razao_credito_bem"] > 80) & (submissao["razao_credito_bem"] < 200)]
print("contratos suspeitos:", len(suspeitos), "| clientes distintos:", suspeitos["id_cliente"].nunique())

In [ ]:
# Passo 1: quantos dos 63 clientes suspeitos aparecem no histórico?
ids_suspeitos = suspeitos["id_cliente"].unique()
hist_susp = emprestimos[emprestimos["id_cliente"].isin(ids_suspeitos)]
print("clientes suspeitos com histórico:", hist_susp["id_cliente"].nunique(), "de", len(ids_suspeitos))
print("contratos desses clientes no histórico:", len(hist_susp))

# Passo 2: o histórico tem o mesmo problema de escala?
# Razão crédito/bem calculada só onde bem e crédito existem e são positivos.
ok = (emprestimos["valor_bem"] > 0) & (emprestimos["valor_credito"] > 0)
emprestimos_razao = np.where(ok, emprestimos["valor_credito"] / emprestimos["valor_bem"], np.nan)
emprestimos["razao_credito_bem"] = emprestimos_razao
print(emprestimos["razao_credito_bem"].describe(percentiles=[0.01, 0.5, 0.99, 0.999]))
print("histórico com razão > 80:", (emprestimos["razao_credito_bem"] > 80).sum())

# Passo 3: para cada cliente suspeito, compara o crédito suspeito com a mediana dos créditos passados.
mediana_hist = (
    hist_susp[hist_susp["valor_credito"] > 0]
    .groupby("id_cliente")["valor_credito"].median()
    .rename("mediana_credito_hist")
)
comparacao = suspeitos[["id_cliente", "valor_credito", "valor_bem"]].merge(
    mediana_hist, on="id_cliente", how="inner"
)
comparacao["credito_sobre_hist"] = comparacao["valor_credito"] / comparacao["mediana_credito_hist"]
print(len(comparacao), "clientes com crédito histórico positivo")
print(comparacao["credito_sobre_hist"].describe())
print(comparacao.sort_values("credito_sobre_hist").head(10))

In [ ]:
# Guarda o valor original antes de qualquer alteração, para auditoria.
submissao["valor_credito_original"] = submissao["valor_credito"]

# Flag: 1 onde a razão crédito/bem está na faixa de erro de escala (80 a 200).
# Linhas sem bem têm razão NaN; a comparação com NaN dá False, então ficam 0.
erro_escala = (submissao["razao_credito_bem"] > 80) & (submissao["razao_credito_bem"] < 200)
submissao["credito_corrigido"] = np.where(erro_escala, 1, 0)

# Corrige dividindo por 100 só onde a flag é 1.
submissao["valor_credito"] = np.where(
    submissao["credito_corrigido"] == 1,
    submissao["valor_credito"] / 100,
    submissao["valor_credito"],
)

# Recalcula a razão com o crédito corrigido.
submissao["razao_credito_bem"] = np.where(
    submissao["valor_bem"] > 0, submissao["valor_credito"] / submissao["valor_bem"], np.nan
)

# Checagens
print("corrigidas:", submissao["credito_corrigido"].sum())
print(submissao["razao_credito_bem"].describe(percentiles=[0.01, 0.99, 0.999]))
print("máximo de valor_credito:", submissao["valor_credito"].max())
print("linhas:", len(submissao))

### Concluídas

| Passo | Coluna | Problema | Decisão | Resultado |
|---|---|---|---|---|
| 2 | `tipo_contrato` | 6 grafias para 3 categorias (caixa e acento, 302 linhas) | Forma normalizada via `padronizar_categoricas` | 3 categorias, sem nulos, 50.400 linhas |
| 3 | `valor_parcela` | 67 zeros com crédito positivo (mín. 101.880), nos 3 tipos de contrato; menor parcela legítima 789; 5 nulos | Zeros viram nulo, sem flag e sem imputação | 72 nulos (0,14%), 0 zeros, 50.400 linhas |
| 4 | `valor_bem` | 2.460 nulos (4,9%), ligados a contratos de menor valor | Manter nulo, sem imputação; flag `valor_bem_ausente` | 2.460 flags = 1, 50.400 linhas |
| 5 | `valor_credito` | 63 linhas com razão crédito/bem entre 80 e 200, ausente no histórico | Dividir por 100; flag `credito_corrigido`; original preservado | 63 corrigidas, 50.400 linhas |

### Em aberto (tratar nesta ordem)

2. `dia_semana_solicitacao`, `data_solicitacao` e `hora_solicitacao`: 51 datas `'2025-13-01'` (provável placeholder), 50 horas negativas, 50 datas em `DD/MM/AAAA`. O dia da semana original está preenchido nas 51 datas inválidas, então a decisão de descartá-lo deve ser reavaliada.
4. Clientes repetidos: 400 clientes com duas linhas; regra de deduplicação a definir antes da separação treino e teste.


### Decisão: `valor_credito` (erro de escala)

Problema: 63 linhas com razão crédito/bem entre 80 e 200 (máx. 152,8); nenhuma entre 5 e 80. Bem e parcela dessas linhas são normais; só o crédito está inflado (ex.: 135.000.000 com bem de 1.350.000).
Validação no histórico: 62 dos 63 clientes têm histórico (266 contratos). Nos 161.422 contratos históricos com bem, a razão crédito/bem tem mediana 1,00 e máximo 6,67; nenhum acima de 80. O padrão não se repete no histórico, o que contraria a hipótese de valores reais.
Regra: dividir por 100 e criar a flag `credito_corrigido`; valor original em `valor_credito_original`.
Resultado: 63 linhas corrigidas, 50.400 linhas preservadas.
Não corrigido: 360 linhas com razão entre 1,5 e 5 e 12 abaixo de 0,5, plausíveis; vão para o cliente.
Limitações: o critério depende de `valor_bem`, então erros nas 2.460 linhas sem bem não são detectados por ele. Após a correção, a razão sobre a mediana histórica do cliente não se concentra em 1 (a mediana estimada fica em torno de 4), sem grupo de controle para comparar.
Para o cliente: o crédito de 63 contratos foi registrado com escala 100x? Razões acima de 1,5 e abaixo de 0,5 são legítimas?